<a href="https://colab.research.google.com/github/praveenpiisc/cp250/blob/main/notebooks/B_sectoral_demand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# B. Standard demand inputs: CAGR to network Loads

Change the standard industry CAGR file, follow its country-to-node calculation, and inspect the resulting electricity Load. Then inspect the separate sector-energy CAGR file.

Run the cells from top to bottom. The examples use one node and 24 hours of synthetic data.
Source links refer to [PyPSA-Earth](https://github.com/pypsa-meets-earth/pypsa-earth/tree/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f).

## Set up the notebook

In a fresh Colab CPU session, uncomment and run the installation cell before importing packages.
If you have already imported numerical packages, restart the runtime after installation.
A local environment with `requirements-colab.txt` can skip this cell.

In [ ]:
# %pip install -q pypsa==1.2.4 linopy==0.8.0 highspy==1.15.1 "numpy<2.5" pandas==2.2.3 "plotly>=5.24.1,<6" PyYAML==6.0.3

In [ ]:
from pathlib import Path

import pypsa
import pandas as pd
import yaml

SOLVER = "highs"
pypsa.options.params.optimize.include_objective_constant = False

The next cell uses local tutorial data or downloads it automatically from GitHub.

In [ ]:
data_dir = Path("tutorial-data")
if not data_dir.exists() and Path("../tutorial-data").exists():
    data_dir = Path("../tutorial-data")
if not data_dir.exists():
    import zipfile
    from urllib.request import urlretrieve

    course_ref = "1f25795897e14d1307003110a6b6587f21155278"
    url = f"https://raw.githubusercontent.com/ljansen-iee/pypsa-meets-earth-lab-2026/{course_ref}/tutorial-data.zip"
    archive_path = Path("tutorial-data.zip")
    urlretrieve(url, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(".")

config = yaml.safe_load((data_dir / "config/config.DEMO_BASE.yaml").read_text())

data_dir = data_dir.resolve()
work_dir = data_dir.parent / "work"

## 1. Read the industry growth inputs

[build_industry_demand](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L2149-L2177)
reads `data/demand/industry_growth_cagr.csv`.

The file has countries as rows and existing subsectors such as `iron and steel` as columns.
Values are annual fractional growth rates: 0.03 is 3% per year. A missing country or
entry uses `DEFAULT`, following [the standard branch](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/build_industry_demand.py#L102-L145).
The teaching base totals use the real generated-file schema but synthetic quantities.

In [ ]:
countries = config["countries"]
investment_year = config["scenario"]["planning_horizons"][0]
no_years = investment_year - config["demand_data"]["base_year"]
industry_growth_path = data_dir / "data/demand/industry_growth_cagr.csv"
industry_growth_cagr = pd.read_csv(industry_growth_path, index_col=0,
                                  keep_default_na=False, na_values=[""])
# Preserve the country code NA; only empty cells represent missing values.
industry_growth_cagr

,chemical and petrochemical,construction,food and tobacco,iron and steel,machinery,mining and quarrying,non-ferrous metals,non-metallic minerals,paper pulp and print,textile and leather,transport equipment,wood and wood products,ammonia,other
country,,,,,,,,,,,,,,
DEFAULT,0.0300,0.0150,0.0200,0.017,0.030,0.0300,0.030,0.0300,0.030,0.0300,0.030,0.0300,0.0300,0.030
MA,0.0300,0.0150,0.0200,0.017,0.030,0.0300,0.030,0.0300,0.030,0.0300,0.030,0.0300,0.0300,0.030
NA,0.0037,-0.0024,-0.0338,NaN,NaN,0.1888,-0.048,-0.0624,NaN,0.0354,NaN,0.0278,0.0037,NaN
US,0.0170,0.0080,0.0094,0.005,0.014,0.0020,0.011,0.0080,0.003,-0.0010,0.014,0.0010,0.0170,0.009


In [ ]:
cagr = industry_growth_cagr.copy()
for country in countries:
    if country not in cagr.index:
        cagr.loc[country] = cagr.loc["DEFAULT"]
    else:
        cagr.loc[country] = cagr.loc[country].fillna(cagr.loc["DEFAULT"])
cagr = cagr.loc[countries]
growth_factors = (1 + cagr) ** no_years
# Expect the DEFAULT iron-and-steel rate of 1.7% over 2019–2035: about 31% growth.
growth_factors

,chemical and petrochemical,construction,food and tobacco,iron and steel,machinery,mining and quarrying,non-ferrous metals,non-metallic minerals,paper pulp and print,textile and leather,transport equipment,wood and wood products,ammonia,other
country,,,,,,,,,,,,,,
ZA,1.604706,1.268986,1.372786,1.30959,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706


In [ ]:
industry_base_totals = pd.read_csv(
    data_dir / "resources/demand/base_industry_totals_2035.csv", index_col=[0, 1],
    keep_default_na=False, na_values=[""])
# Carrier rows are MWh/year; process emissions, where present, are tCO2/year.
# The base producer converts TWh to MWh before writing this resource.
industry_base_totals.loc["ZA", ["iron and steel"]]

,iron and steel
carrier,
electricity,17520.0
gas,0.0
coal,35040.0
oil,0.0
hydrogen,0.0
ammonia,0.0
biomass,0.0
low-temperature heat,0.0
process emissions,0.0


[Base industry energy-unit conversion](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/build_base_industry_totals.py#L24-L83).
The year in `base_industry_totals_2035.csv` is the workflow target label; the growth interval
still starts from `demand_data.base_year`. The same growth factor multiplies each carrier total within a subsector, preserving the base carrier mix.

## 2. Allocate projected activity and multiply by base carrier totals

Use the same `dist_keys`, `production_tom`, `nodal_production_tom` and dot-product pattern as
[country_to_nodal](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/build_industry_demand.py#L25-L49) and
[nodal demand calculation](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/build_industry_demand.py#L373-L394).
There is one teaching node, so every sector's country share is one. In a clustered
country model those shares distribute demand over several nodes; they do not create extra demand.

In [ ]:
dist_keys = pd.read_csv(
    data_dir / "resources/demand/industrial_distribution_key_elec_s_1_2035.csv", index_col=0,
    keep_default_na=False, na_values=[""])
# Activity is normalised to 1 in the base year; growth and spatial shares are fractions.
production_base = pd.DataFrame(1.0, index=cagr.index, columns=cagr.columns)
production_tom = production_base * growth_factors
nodal_production_tom = pd.DataFrame(index=dist_keys.index, columns=cagr.columns, dtype=float)
for country in countries:
    buses = dist_keys.index[dist_keys.country == country]
    for sector in cagr.columns:
        mapping = sector if sector in dist_keys.columns and dist_keys[sector].sum() != 0 else "gdp"
        nodal_production_tom.loc[buses, sector] = production_tom.at[country, sector] * dist_keys.loc[buses, mapping]
nodal_production_tom

,chemical and petrochemical,construction,food and tobacco,iron and steel,machinery,mining and quarrying,non-ferrous metals,non-metallic minerals,paper pulp and print,textile and leather,transport equipment,wood and wood products,ammonia,other
name,,,,,,,,,,,,,,
ZA0,1.604706,1.268986,1.372786,1.30959,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706,1.604706


In [ ]:
# Subsector activity × carrier-specific base energy gives annual nodal energy.
country = "ZA"
buses = dist_keys.index[dist_keys.country == country]
nodal_df = nodal_production_tom.loc[buses].dot(industry_base_totals.loc[country].T)
nodal_df = nodal_df.rename(columns={"biomass": "solid biomass"})
nodal_df.index.name = "MWh/a (tCO2/a)"
# Expect the electricity and coal rows to grow at the same steel CAGR.
# Hydrogen starts at zero in these base data and remains zero.
nodal_df

carrier,electricity,gas,coal,oil,hydrogen,ammonia,solid biomass,low-temperature heat,process emissions
MWh/a (tCO2/a),,,,,,,,,
ZA0,22944.011338,0.0,45888.022676,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Increase iron-and-steel CAGR to 3%

Change the iron-and-steel **input rate** to 3%, retaining the other countries and subsectors.
Write the modified file under `work/data/demand/` and compare its demand projection
with the original rate.

In [ ]:
industry_growth_high = industry_growth_cagr.copy()
industry_growth_high.loc["ZA"] = cagr.loc["ZA"]
industry_growth_high.loc["ZA", "iron and steel"] = 0.03
work = work_dir
(work / "data/demand").mkdir(parents=True, exist_ok=True)
industry_growth_high.to_csv(work / "data/demand/industry_growth_cagr.csv")
industry_growth_high.loc[["DEFAULT", "ZA"], ["iron and steel"]]

,iron and steel
country,
DEFAULT,0.017
ZA,0.030


In [ ]:
cagr_high = industry_growth_high.loc[countries]
production_tom_high = (1 + cagr_high) ** no_years
nodal_production_high = nodal_production_tom.copy()
nodal_production_high.loc[buses] = dist_keys.loc[buses, cagr.columns].mul(production_tom_high.loc[country], axis=1)
nodal_df_high = nodal_production_high.loc[buses].dot(industry_base_totals.loc[country].T)
nodal_df_high = nodal_df_high.rename(columns={"biomass": "solid biomass"})
pd.concat({"baseline MWh/year": nodal_df, "higher CAGR MWh/year": nodal_df_high}, axis=1)

baseline MWh/year                                           \
carrier       electricity  gas          coal  oil hydrogen ammonia   
ZA0          22944.011338  0.0  45888.022676  0.0      0.0     0.0   

                                                              \
carrier solid biomass low-temperature heat process emissions   
ZA0               0.0                  0.0               0.0   

        higher CAGR MWh/year                                           \
carrier          electricity  gas          coal  oil hydrogen ammonia   
ZA0             28114.456813  0.0  56228.913626  0.0      0.0     0.0   

                                                              
carrier solid biomass low-temperature heat process emissions  
ZA0               0.0                  0.0               0.0

## 4. Inspect the change at the network Load

[Generated industry CSV input](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L1387-L1393) feeds
[the industry electricity Load](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L2027-L2036).
The name is `ZA0 industry electricity`, with carrier `industry electricity` and annual MWh/8760.
Keep that divisor for this 24-hour subset: dividing annual demand by 24 would concentrate
an entire year's energy into one day.

In [ ]:
n = pypsa.Network(data_dir / "networks/elec_s_1_ec_lcopt_1h_sector.nc")

# Carrier colours are plotting metadata, shared with notebook A.
n.carriers.update(pd.DataFrame({"color": {
    "AC": "#a6bbc8",
    "solar": "#fdb913",
    "onwind": "#005b7f",
    "OCGT": "#d3c7ae",
    "industry electricity": "#a6bbc8",
    "battery": "#836bad",
    "H2": "#179c7d",
    "H2 Electrolysis": "#179c7d",
    "H2 export": "#4CC2A6",
    "H2 Store Tank": "#179c7d",
    "water": "#39c1cd",
    "water supply": "#39c1cd",
}}))
before = n.loads.loc[["ZA0 industry electricity"], ["p_set"]].rename(columns={"p_set": "baseline MW"})
n.loads.loc["ZA0 industry electricity", "p_set"] = nodal_df_high.at["ZA0", "electricity"] / 8760
before["higher CAGR MW"] = n.loads.loc["ZA0 industry electricity", "p_set"]
before

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)


INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads, storage_units, stores


,baseline MW,higher CAGR MW
name,,
ZA0 industry electricity,2.619179,3.209413


In [ ]:
n.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Expect a larger electricity requirement with the same generators and flexibility capacities.
# This change scales the existing carrier mix; it does not turn coal demand into hydrogen demand.

INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - threads: 1
 - log_to_console: False


INFO:linopy.io: Writing time: 0.24s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 264 primals, 624 duals
Objective: 6.48e+03
Solver: highs
Runtime: 0.01s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


In [ ]:
# Snapshot weights integrate this day to MWh; these are not the annual input totals.
n.statistics.energy_balance(bus_carrier="AC", groupby=["carrier"], nice_names=False)

component    carrier             
Generator    OCGT                    55.93342
             onwind                  44.80000
             solar                   91.14905
Link         H2 Electrolysis        -34.28571
Load         AC                     -79.18792
             industry electricity   -77.02591
StorageUnit  battery                 -1.38293
dtype: float64

## 5. Inspect the other CAGR file

`data/demand/growth_factors_cagr.csv` controls the separate national sector-energy pathway,
not the industrial-subsector growth calculation. Its partner is `efficiency_gains_cagr.csv`.
[prepare_energy_totals inputs](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L1617-L1632);
[fallback, growth and energy multiplication](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_energy_totals.py#L50-L96).
Road transport uses separate efficiency columns for combustion, fuel-cell and electric
vehicles. The [road calculation](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_energy_totals.py#L246-L257)
weights those factors by the vehicle shares before applying road growth. Here the
electric share is 100%; keep it fixed when comparing the two growth rates.
The example base totals are in MWh/year; the full workflow's national energy totals
use TWh. Growth and efficiency factors are dimensionless.

In [ ]:
base_energy_totals = pd.read_csv(data_dir / "resources/energy_totals_base.csv", index_col=0,
                                keep_default_na=False, na_values=[""])
growth_factors_cagr = pd.read_csv(data_dir / "data/demand/growth_factors_cagr.csv", index_col=0,
                                 keep_default_na=False, na_values=[""])
efficiency_gains_cagr = pd.read_csv(data_dir / "data/demand/efficiency_gains_cagr.csv", index_col=0,
                                   keep_default_na=False, na_values=[""])
for country in countries:
    for table in (growth_factors_cagr, efficiency_gains_cagr):
        if country not in table.index:
            table.loc[country] = table.loc["DEFAULT"]
        else:
            table.loc[country] = table.loc[country].fillna(table.loc["DEFAULT"])
growth_factors = (1 + growth_factors_cagr.loc[countries]) ** no_years
efficiency_gains = (1 + efficiency_gains_cagr.loc[countries]) ** no_years
energy_columns = base_energy_totals.columns.intersection(efficiency_gains.columns)
energy_totals = (base_energy_totals[energy_columns]
                 * efficiency_gains[energy_columns] * growth_factors[energy_columns])

electric_share = config["sector"]["land_transport_electric_share"]
fuel_cell_share = config["sector"]["land_transport_fuel_cell_share"]
road_efficiency = ((1 - electric_share - fuel_cell_share) * efficiency_gains["total road ice"]
                   + fuel_cell_share * efficiency_gains["total road fcev"]
                   + electric_share * efficiency_gains["total road ev"])
energy_totals["total road"] = (base_energy_totals["total road"]
                               * road_efficiency * growth_factors["total road"])
# Road efficiency is weighted by the combustion, fuel-cell and electric shares.
energy_totals[["electricity residential", "total road"]]

,electricity residential,total road
country,,
ZA,59893.961269,3171.062925


In [ ]:
growth_factors_high = growth_factors_cagr.copy()
growth_factors_high.loc["ZA", "total road"] = 0.03
energy_totals_high = energy_totals.copy()
energy_totals_high["total road"] = (base_energy_totals["total road"] * road_efficiency
                                    * (1 + growth_factors_high.loc[countries, "total road"]) ** no_years)
growth_factors_high.to_csv(work / "data/demand/growth_factors_cagr.csv")
road_sensitivity = pd.DataFrame({
    "baseline road [MWh/year]": energy_totals["total road"],
    "3% road growth [MWh/year]": energy_totals_high["total road"],
})
# Expect about 3,171 versus 4,080 MWh/year with the same vehicle shares and efficiency assumptions.
road_sensitivity

,baseline road [MWh/year],3% road growth [MWh/year]
country,,
ZA,3171.062925,4080.172143


## From this notebook to a country workflow

Copy the edited CAGR file to your PyPSA-Earth checkout.
Both CAGR files are declared Snakemake inputs, so their changes are tracked.
Keep baseline results in their scenario directory before replacing the input for another run.

Trace the changed file through `build_industry_demand` or `prepare_energy_totals`,
then [prepare_sector_network](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L1339-L1425),
[add_export](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L1439-L1464) and the workflow solve rule.
The full model also adjusts the electricity accounting in other sector functions;
this notebook isolates the industrial Load rather than reproducing that full accounting.

### Interpretation and recap

Discuss these questions using the tables and plots above. Use them for the joint A–C recap and quiz after the break.

1. What is the difference between an annual growth rate and the compounded growth factor? Which base year and planning horizon determine that factor?
2. Which file changes steel growth, and which changes road activity? Trace an edited rate through the generated annual demand to the receiving Load, explaining why annual MWh are divided by 8,760 in this day-long example.
3. Why does increasing steel CAGR not create hydrogen use when the base entry is zero? What additional demand or process assumptions would represent a different steel route?

**Approach critique:** Which structural changes are omitted when one growth factor scales the existing carrier mix?
